# Crogrammar — trening ByT5 hrvatskog GEC modela

Ovaj notebook fine-tuna `google/byt5-small` na parovima (pogrešna, ispravna) rečenica.
Predviđen za Google Colab (Tesla T4) ili Kaggle. Checkpointi se spremaju na Google
Drive radi nastavljivosti (Colab prekida sesije).

Licence podataka: ispravi.me = CC BY-NC-SA 4.0 (nekomercijalno), hr500k = CC BY-SA 4.0.

## 1. Kloniranje repozitorija i instalacija (s [train] extras)

In [ ]:
!git clone https://github.com/<tvoj-korisnik>/crogrammar.git
%cd crogrammar
!pip install -e ".[train]"

## 2. Montiranje Google Drive-a (za nastavljive checkpointe)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Preuzimanje izvora (hr500k + hunspell-hr rječnik)

In [ ]:
from crogrammar.data.download import download_hr500k, download_hunspell_dic
download_hr500k('data/raw')
download_hunspell_dic('data/raw')

## 4. Izgradnja dataseta -> data/processed/*.jsonl

Učitaj čiste rečenice iz hr500k, izgradi dijakritički confusion set iz hunspell-hr
rječnika (~15k riječi), generiraj parove i podijeli na train/dev/test.

In [ ]:
import gzip
from pathlib import Path
from crogrammar.data.conllu import parse_conllu, sentence_text
from crogrammar.data.confusion import load_confusion_from_dic, load_wordset_from_dic
from crogrammar.data.build_dataset import make_pairs, split_pairs, write_jsonl

clean = []
for gz in Path('data/raw/hr500k').glob('*.conllu.gz'):
    with gzip.open(gz, 'rt', encoding='utf-8') as f:
        for sent in parse_conllu(f.read()):
            clean.append(sentence_text(sent))

confusion = load_confusion_from_dic('data/raw/hr_HR.dic')
real_words = load_wordset_from_dic('data/raw/hr_HR.dic')
print('confusion set:', len(confusion), 'rijeci |', 'rjecnik:', len(real_words), 'rijeci')
pairs = make_pairs(clean, confusion, seed=42, variants=3, real_words=real_words)
train, dev, test = split_pairs(pairs, dev_frac=0.05, test_frac=0.05, seed=42)
write_jsonl(train, 'data/processed/train.jsonl')
write_jsonl(dev, 'data/processed/dev.jsonl')
write_jsonl(test, 'data/processed/test.jsonl')
print(len(train), len(dev), len(test))

## 5. Trening (nastavljiv iz checkpointa na Drive-u)

Prvi trening kreće od nule; ako te Colab prekine, ponovno pokreni ovu ćeliju
i nastavit će od zadnjeg checkpointa na Drive-u.

In [ ]:
from crogrammar.train.config import TrainConfig
from crogrammar.train.train import train

cfg = TrainConfig(output_dir='/content/drive/MyDrive/crogrammar/byt5-hr-gec')
train(cfg, resume_from_checkpoint=True)

## 6. Evaluacija na test setu (korpusni GLEU)

In [ ]:
from crogrammar.inference.gec import CroatianGEC
from crogrammar.eval.run_eval import evaluate_pairs, load_jsonl

gec = CroatianGEC(model_path='/content/drive/MyDrive/crogrammar/byt5-hr-gec')
test_pairs = load_jsonl('data/processed/test.jsonl')
score = evaluate_pairs(test_pairs, gec._generate)
print('GLEU:', score['gleu'], 'n:', score['n'])

## 7. Zapis METADATA.json uz model (verzija, GLEU, licenca)

In [ ]:
import json, datetime
meta = {
    'version': '0.1.0',
    'date': datetime.date.today().isoformat(),
    'base_model': cfg.base_model,
    'gleu_test': score['gleu'],
    'license': 'CC BY-NC-SA 4.0 (nekomercijalno)',
    'attributions': [
        'ispravi.me dataset: Gledec et al. 2023, CC BY-NC-SA 4.0',
        'hr500k: Ljubesic & Samardzic, CLARIN.SI, CC BY-SA 4.0',
    ],
}
with open(cfg.output_dir + '/METADATA.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(meta)